# 23 — Per-Category Feature-Shift Matrix, Cross-Dataset (reviewer A)

**Dijalankan di SageMaker.** Menjawab reviewer JISA (A): matriks pergeseran fitur
untuk kategori serangan utama, **lintas-dataset** CIC vs UNSW. CIC-IDS2018 CSV
(`file_100.csv`) menyimpan label kategori asli (DoS Hulk/GoldenEye, DDoS HOIC/LOIC,
Brute-Force FTP/SSH, Botnet, Infiltration, Web); UNSW punya `attack_cat`. Kita
petakan keduanya ke **grup kategori umum**, lalu untuk tiap kategori ukur jarak
Wasserstein $W_1$ ke-9 fitur SFM terhadap **benign** (z-space per-dataset).

Tujuan: buktikan kuantitatif bahwa (i) kategori serangan berbeda menggeser fitur
berbeda, dan (ii) kategori sepadan (mis. DoS) bergeser — tapi — secara berbeda di
CIC vs UNSW, sehingga kalibrasi global tunggal tak cukup untuk few-shot multi-class.

In [ ]:
import importlib.util as u, sys, subprocess
need=[m for m in ('scipy','pandas','numpy','matplotlib','boto3') if u.find_spec(m) is None]
if need: subprocess.run([sys.executable,'-m','pip','install','-q',*need],check=True)
print('setup ok' if not need else f'installed {need}')
print('=== SEL 0 (setup) SELESAI ===')

In [ ]:
import os, json, glob, datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
from scipy.stats import wasserstein_distance
plt.rcParams.update({'figure.dpi':120,'font.size':9})
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717')
S3_PREFIX='unsw-far'; REGION=os.environ.get('AWS_REGION','ap-southeast-1')
OUTDIR='catshift_out'; os.makedirs(OUTDIR,exist_ok=True)
CANON=['duration','fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','fwd_mean','bwd_mean','src_load','dst_load']
MIN_N=200  # minimal sampel per kategori agar W1 stabil
RESULTS={'generated':datetime.datetime.utcnow().isoformat()+'Z','features':CANON}
def savefig(n): p=os.path.join(OUTDIR,n); plt.savefig(p,bbox_inches='tight'); plt.close(); print(' saved',p); return p
def first(paths):
    for p in paths:
        h=sorted(glob.glob(p))
        if h: return h[0]
    return None
print('=== SEL 1 (config) SELESAI ===')

## 2. Pemetaan label -> grup kategori umum

In [ ]:
# Grup umum yang dipakai untuk membandingkan lintas-dataset.
def map_cic(lbl):
    s=str(lbl).strip().lower()
    if s in ('benign','normal'): return 'Benign'
    if s.startswith('ddos') or 'loic' in s or 'hoic' in s: return 'DDoS'
    if s.startswith('dos'): return 'DoS'
    if 'bruteforce' in s or 'brute force' in s or 'ftp-brute' in s or 'ssh-brute' in s: return 'BruteForce'
    if s=='bot' or 'botnet' in s: return 'Botnet'
    if 'infil' in s: return 'Infiltration'
    if 'web' in s or 'xss' in s or 'sql' in s: return 'Web'
    return 'Other'
def map_uns(lbl):
    s=str(lbl).strip().lower()
    if s in ('normal','benign',''): return 'Benign'
    if s=='dos': return 'DoS'
    if s=='exploits': return 'Exploits'
    if s=='fuzzers': return 'Fuzzers'
    if s=='generic': return 'Generic'
    if s=='reconnaissance': return 'Recon'
    if s=='backdoor' or s=='backdoors': return 'Backdoor'
    if s=='shellcode': return 'Shellcode'
    if s=='worms': return 'Worms'
    if s=='analysis': return 'Analysis'
    return 'Other'
print('=== SEL 2 (pemetaan kategori) SELESAI ===')

## 3. Muat CIC (file_100.csv) + 9 fitur + kategori

In [ ]:
CIC_CSV=first(['../../CICDDoS2018/data/file_100.csv','../../CICDDoS2018/data/file_*.csv'])
cic=None
if CIC_CSV:
    c=pd.read_csv(CIC_CSV, low_memory=False)
    c.columns=c.columns.str.strip()
    cmap={'duration':'Flow Duration','fwd_pkts':'Tot Fwd Pkts','bwd_pkts':'Tot Bwd Pkts','fwd_bytes':'TotLen Fwd Pkts',
          'bwd_bytes':'TotLen Bwd Pkts','fwd_mean':'Fwd Pkt Len Mean','bwd_mean':'Bwd Pkt Len Mean','src_load':'Flow Byts/s','dst_load':'Bwd Pkts/s'}
    lc=[x for x in c.columns if x.lower()=='label']; LAB=lc[0] if lc else c.columns[-1]
    if all(v in c.columns for v in cmap.values()):
        cic=pd.DataFrame({k:pd.to_numeric(c[cmap[k]],errors='coerce') for k in CANON})
        cic['cat']=c[LAB].map(map_cic)
        cic=cic.replace([np.inf,-np.inf],np.nan).dropna()
        cic=cic[cic['cat']!='Other']
        print('CIC rows:',len(cic),'| kategori:',cic['cat'].value_counts().to_dict())
    else:
        print('CIC: kolom fitur tak lengkap; dilewati. Ada:',[v for v in cmap.values() if v in c.columns])
else:
    print('file_100.csv tidak ditemukan')
print('=== SEL 3 (muat CIC) SELESAI ===')

## 4. Muat UNSW + 9 fitur (satuan dikoreksi) + kategori

In [ ]:
UNS_CSV=first(['../data/UNSW_NB15_training-set.csv','../data/UNSW_NB15_*set.csv'])
uns=None
if UNS_CSV:
    u2=pd.read_csv(UNS_CSV)
    need=['dur','spkts','dpkts','sbytes','dbytes','smean','dmean','sload','dload','attack_cat']
    if all(x in u2.columns for x in need):
        uns=pd.DataFrame({'duration':pd.to_numeric(u2['dur'],errors='coerce')*1e6,'fwd_pkts':u2['spkts'],'bwd_pkts':u2['dpkts'],
                          'fwd_bytes':u2['sbytes'],'bwd_bytes':u2['dbytes'],'fwd_mean':u2['smean'],'bwd_mean':u2['dmean'],
                          'src_load':pd.to_numeric(u2['sload'],errors='coerce')/8.0,
                          'dst_load':pd.to_numeric(u2['dpkts'],errors='coerce')/pd.to_numeric(u2['dur'],errors='coerce').replace(0,np.nan)})
        uns['cat']=u2['attack_cat'].fillna('Normal').map(map_uns)
        uns=uns.replace([np.inf,-np.inf],np.nan).dropna(); uns=uns[uns['cat']!='Other']
        print('UNSW rows:',len(uns),'| kategori:',uns['cat'].value_counts().to_dict())
    else:
        print('UNSW: kolom kurang:',[x for x in need if x not in u2.columns])
else:
    print('UNSW csv tak ditemukan')
print('=== SEL 4 (muat UNSW) SELESAI ===')

## 5. Matriks $W_1$ per-kategori vs benign (per-dataset, z-space)

In [ ]:
def cat_matrix(df, name):
    if df is None or len(df)==0: return None
    mu=df[CANON].mean(); sd=df[CANON].std().replace(0,1)
    ben=df[df['cat']=='Benign'][CANON]
    if len(ben)==0: print(name,'tak ada benign'); return None
    benz=(ben-mu)/sd
    rows=[]
    for c,n in df['cat'].value_counts().items():
        if c=='Benign' or n<MIN_N: continue
        z=(df[df['cat']==c][CANON]-mu)/sd
        r={'dataset':name,'category':c,'n':int(n)}
        for f in CANON: r[f]=round(float(wasserstein_distance(z[f].values, benz[f].values)),4)
        r['MEAN']=round(float(np.mean([r[f] for f in CANON])),4)
        rows.append(r)
    return pd.DataFrame(rows)

mc=cat_matrix(cic,'CIC'); mu=cat_matrix(uns,'UNSW')
mat=pd.concat([x for x in [mc,mu] if x is not None], ignore_index=True)
import IPython.display as ipd; ipd.display(mat[['dataset','category','n','MEAN']+CANON])
RESULTS['per_category_w1']=mat.to_dict(orient='records')
print('=== SEL 5 (matriks W1 per-kategori) SELESAI ===')

## 6. Heatmap (kategori x fitur), dipisah per dataset

In [ ]:
lab=[f"{r.dataset}:{r.category}" for r in mat.itertuples()]
M=mat[CANON].values
fig,ax=plt.subplots(figsize=(9,0.7+0.5*len(mat)))
im=ax.imshow(M,aspect='auto',cmap='YlOrRd')
ax.set_xticks(range(len(CANON))); ax.set_xticklabels(CANON,rotation=45,ha='right')
ax.set_yticks(range(len(lab))); ax.set_yticklabels(lab)
for i in range(M.shape[0]):
    for j in range(M.shape[1]): ax.text(j,i,f'{M[i,j]:.2f}',ha='center',va='center',fontsize=7)
fig.colorbar(im,ax=ax,label='$W_1$ vs benign')
ax.set_title('Per-category feature-shift matrix, cross-dataset ($W_1$ vs benign)')
plt.tight_layout(); savefig('category_shift_matrix.png')
print('=== SEL 6 (heatmap) SELESAI ===')

## 7. Simpan + UPLOAD S3

In [ ]:
mat.to_csv(os.path.join(OUTDIR,'category_shift_matrix.csv'),index=False)
jp=os.path.join(OUTDIR,'category_shift_results.json')
with open(jp,'w') as f: json.dump(RESULTS,f,indent=2)
print('tersimpan',jp)
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.endswith(('.json','.png','.csv')): s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'{S3_PREFIX}/catshift/{fn}'); up+=1
    print(f'upload {up} artefak -> s3://{S3_BUCKET}/{S3_PREFIX}/catshift/')
except Exception as e: print('upload gagal:',e)
print('=== SEL 7 (simpan + upload) SELESAI ===')
print('SEMUA SELESAI. Beri tahu asisten -> unduh s3://%s/%s/catshift/ untuk analisis.'%(S3_BUCKET,S3_PREFIX))